Assigment 2
```
# COSC5437002 - Neural Networks and Deep Learning
# Prof. Syed Muhammad Danish
# Algoma University
# Department of Computer Science and Mathematics
# Brampton, Ontario, Canada
# Date: 2025
```

In [1]:
import matplotlib.pyplot as plt
import torchvision
import numpy as np
from torchvision import transforms
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init

In [2]:
transform = transforms.Compose([ # Transform: flatten the image to a vector
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # normalize to [-1, 1]
])

train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
print(f"Total training samples: {len(train_dataset)}")
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
print(f"Total test samples: {len(test_dataset)}")

# ~> Setting batch size as 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

100%|██████████| 169M/169M [00:05<00:00, 28.3MB/s]


Total training samples: 50000
Total test samples: 10000
cuda


In [ ]:
class DenseANN_BatchNorm(nn.Module): # Model definition
    def __init__(self):
        super(DenseANN_BatchNorm, self).__init__()
        self.flatten = nn.Flatten() # Make onde dimention

        self.fc1 = nn.Linear(32*32*3, 2048)
        # ~> Batch Normalization to normalize the weights
        self.bn1 = nn.BatchNorm1d(2048)
        # ~> LeakyReLu let negative values, helping to mitigate the vanishing gradient problem
        self.ac1 = nn.LeakyReLU(negative_slope=0.01)
         # ~> Dropout reduces overfitting. Dropout rate is 40%
        self.drop1 = nn.Dropout(p=0.4)

        self.fc2 = nn.Linear(2048, 1024)
        self.bn2 = nn.BatchNorm1d(1024)
        self.ac2 = nn.LeakyReLU(negative_slope=0.01)
        self.drop2 = nn.Dropout(p=0.4)

        # ~> Adding another hidden layer for increased depth
        self.fc3 = nn.Linear(1024, 512)
        self.bn3 = nn.BatchNorm1d(512)
        self.ac3 = nn.LeakyReLU(negative_slope=0.01)

        self.fc4 = nn.Linear(512, 100) # 100 classes for CIFAR-100

    def forward(self, x):
        x = self.flatten(x)

        x = self.fc1(x)
        x = self.bn1(x)
        x = self.ac1(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.ac2(x)
        x = self.drop2(x)

        # third hidden layer:
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.ac3(x)

        x = self.fc4(x)

        return x

# Instantiate and send to device
model = DenseANN_BatchNorm().to(device)

# ~> Loss: Cross Entropy is for classification models
criterion = nn.CrossEntropyLoss()
# ~> Optimizer: Adam is a combination of Momentum and RMSProp (Considered the best in mostly cases); Low learning rate and weight decay are great parameters
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

In [6]:
# Training loop with weight visualization
numEpoch = 50
for epoch in range(numEpoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    print(f"\nEpoch [{epoch+1}/{numEpoch}] Loss: {total_loss:.4f} Accuracy: {correct/total*100:.2f}%")


# Test accuracy
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"\nFinal Test Accuracy: {correct / total * 100:.2f}%")


Epoch [1/50] Loss: 5929.5247 Accuracy: 11.92%

Epoch [2/50] Loss: 5430.9534 Accuracy: 17.11%

Epoch [3/50] Loss: 5239.5268 Accuracy: 19.33%

Epoch [4/50] Loss: 5116.3748 Accuracy: 21.08%

Epoch [5/50] Loss: 5014.5714 Accuracy: 21.86%

Epoch [6/50] Loss: 4936.1631 Accuracy: 22.97%

Epoch [7/50] Loss: 4862.9224 Accuracy: 23.78%

Epoch [8/50] Loss: 4806.0261 Accuracy: 24.45%

Epoch [9/50] Loss: 4744.8194 Accuracy: 25.14%

Epoch [10/50] Loss: 4688.9554 Accuracy: 25.73%

Epoch [11/50] Loss: 4649.5295 Accuracy: 26.43%

Epoch [12/50] Loss: 4598.8932 Accuracy: 27.05%

Epoch [13/50] Loss: 4558.3892 Accuracy: 27.39%

Epoch [14/50] Loss: 4518.2652 Accuracy: 27.77%

Epoch [15/50] Loss: 4483.1734 Accuracy: 28.28%

Epoch [16/50] Loss: 4450.8257 Accuracy: 28.63%

Epoch [17/50] Loss: 4409.2774 Accuracy: 28.88%

Epoch [18/50] Loss: 4384.7470 Accuracy: 29.51%

Epoch [19/50] Loss: 4349.2256 Accuracy: 29.76%

Epoch [20/50] Loss: 4316.7957 Accuracy: 30.35%

Epoch [21/50] Loss: 4292.4999 Accuracy: 30.33%

